# BB84

In [1]:
import numpy as np
from qutip import basis, tensor, qeye, bell_state

rng = np.random.default_rng(42)

ket0, ket1 = basis(2, 0), basis(2, 1)
ketp, ketm = (ket0 + ket1).unit(), (ket0 - ket1).unit()
BASES = {"Z": (ket0, ket1), "X": (ketp, ketm)}

Alice sends a random bit in a random basis; Bob measures in a random basis; they
keep the rounds where the bases matched and count errors.

Eve measures in a random basis and resends. She picks the wrong basis half the
time, and when she does Bob's result is random — so the error rate is 25%.

In [2]:
def measure(state, basis_name):
    kets = BASES[basis_name]
    p = np.array([abs(k.overlap(state)) ** 2 for k in kets])
    out = rng.choice(2, p=p / p.sum())
    return out, kets[out]          # state collapses onto the ket found


In [3]:
def bb84(n, eve):
    errors = sifted = 0
    for _ in range(n):
        bit = rng.integers(2)
        a_basis, b_basis = rng.choice(["Z", "X"]), rng.choice(["Z", "X"])
        psi = BASES[a_basis][bit]                       # Alice prepares

        if eve:
            _, psi = measure(psi, rng.choice(["Z", "X"]))   # intercept-resend

        out, _ = measure(psi, b_basis)                  # Bob measures
        if a_basis == b_basis:                          # sifting
            sifted += 1
            errors += out != bit
    return errors / sifted

In [4]:
print(f"BB84   no Eve:  QBER = {bb84(3000, eve=False):.1%}")
print(f"BB84  with Eve: QBER = {bb84(3000, eve=True):.1%}")

BB84   no Eve:  QBER = 0.0%
BB84  with Eve: QBER = 26.3%
